# Computer Vision Coursework Submission (INM460)

**Student name, ID and cohort:** Bhuvanesi Barua (240067790) - PG


# Notebook Setup
In this section you should include all the code cells required to test your coursework submission. Specifically:

### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Define Local Path

In the next cell you should assign to the variable `GOOGLE_DRIVE_PATH_AFTER_MYDRIVE` the relative path of this folder in your Google Drive.

**IMPORTANT:** you have to make sure that **all the files required to test your functions are loaded using this variable** (as was the case for all lab tutorials). In other words, do not use in the notebook any absolute paths. This will ensure that the markers can run your functions. Also, **do not use** the magic command `%cd` to change directory.



In [ ]:
import os

# TODO: Fill in the Google Drive path where you uploaded the CW_folder_PG
# Example: GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'Colab Notebooks/Computer Vision/CW_folder_PG'

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'CW_Folder_PG'
GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print(os.listdir(GOOGLE_DRIVE_PATH))

['test_function.ipynb', 'CW_Dataset', 'Models', 'Processed_Data', 'Personal_Video', 'X_test.npy', 'y_test.npy', 'CVmain.ipynb']


### Load packages

In the next cell you should load all the packages required to test your function.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from joblib import dump, load
from tensorflow.keras.models import load_model

### Load models

In the next cell you should load your best performing model (this might consist of more than one file). Avoid to load it within `AgeDetection` to avoid having to reload it each time.

In [ ]:
model_path = os.path.join(GOOGLE_DRIVE_PATH, "Models/cnn_model.h5")
model = load_model(model_path)

# Test AgeDetection

This section should allow to test the `AgeDetection` function. First, add cells with the code needed to load the necessary subroutines to make `AgeDetection` work.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from tensorflow.keras.models import load_model
from IPython.display import HTML

# Load face detector (outside function is fine)
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# Load model (make sure path is correct in your notebook)
# model = load_model(model_path)


def AgeDetection(video_path):

    cap = cv2.VideoCapture(video_path)

    frames = []
    frame_count = 0

    labels = ["Child", "Young", "Middle", "Senior"]

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Process every 10th frame (as guideline suggests)
        if frame_count % 10 == 0:

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.3, 5)

            for (x, y, w, h) in faces:
                face = frame[y:y+h, x:x+w]

                # Preprocess for CNN
                face = cv2.resize(face, (128, 128))
                face = face / 255.0
                face = np.expand_dims(face, axis=0)

                pred = model.predict(face, verbose=0)
                label = np.argmax(pred)

                # Draw bounding box
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

                # Draw label (bigger + readable)
                cv2.putText(
                    frame,
                    labels[label],
                    (x, max(y-10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (0,255,0),
                    3
                )

            # Convert BGR → RGB for matplotlib
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_rgb)

        frame_count += 1

    cap.release()

    # Create animation
    fig = plt.figure()
    im = plt.imshow(frames[0])

    def update(i):
        im.set_array(frames[i])
        return [im]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=200
    )

    plt.axis('off')
    plt.close()  # prevents duplicate output

    return HTML(anim.to_jshtml())

Then, make a call to the `AgeDetection` function to see what results it produces.

In [ ]:
# Syntax for the next function is the following:
#
# AgeDetection(path_to_test)

path_to_test = os.path.join(GOOGLE_DRIVE_PATH, "Personal_Video/video_final_h264.mp4")
AgeDetection(path_to_test)

Output hidden; open in https://colab.research.google.com to view.